# PromptEHR: Demographic-Conditioned Synthetic EHR Generation

_Last updated: 2026-03-03 · commit `394e128e`_

Train **PromptEHR** on your MIMIC-III data and generate synthetic patients whose demographic distributions mirror the real population.

## What You'll Need

1. **MIMIC-III Access** (or run in Demo Mode without it). Download 3 files from PhysioNet:
   - `PATIENTS.csv` — patient demographics (date of birth, gender)
   - `ADMISSIONS.csv` — hospital admission records (timestamps)
   - `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

2. **Google Colab** (or local environment): Free tier works; GPU recommended.

> **Demo Mode**: No MIMIC-III? Set `PRESET = "demo"` and skip the file upload step. The notebook runs the full pipeline with synthetic stand-in data.

## What You'll Get

- A trained PromptEHR model conditioned on patient age and gender
- Synthetic patients whose age/gender distributions mirror the MIMIC-III population
- `synthetic_patients.csv` — flat `SUBJECT_ID, VISIT_NUM, ICD9_CODE` records
- `synthetic_patients.json` — nested visit records for PyHealth downstream tasks
- `quality_report.json` — statistics for automated evaluation and CI

## How Long It Takes

| Preset | Epochs | Time (T4 GPU) | Use case |
|--------|--------|----------------|----------|
| `"demo"` | 5 | ~30–45 min | First run, CI smoke test |
| `"production"` | 20 | ~3–5 hrs | Publication-quality results |

## What Makes PromptEHR Different from HALO

Unlike HALO (which generates patients from a shared unconditional distribution), **PromptEHR conditions generation on patient demographics**. It uses a BART Seq2Seq Transformer with learned "prompt" vectors — one per demographic feature — prepended to the encoder input. During training, the model learns that older male patients tend to have different diagnosis patterns than young female patients. During generation, demographics are sampled from the real training distribution, so the synthetic cohort's age/gender profile automatically mirrors MIMIC-III.

This matters clinically: synthetic datasets used for fairness research or subgroup analysis must preserve demographic distributions. PromptEHR provides this guarantee by design.

**Reference**: Wang et al., "PromptEHR: Conditional Electronic Healthcare Records Generation with Prompt Learning." EMNLP 2023. https://arxiv.org/abs/2211.01761

---
# 1. Setup & Installation

In [ ]:
import subprocess
import sys

# Install PyHealth from GitHub — force-reinstall ensures Colab never uses a stale cached build
FORK = 'jalengg'
BRANCH = 'promptehr-pr-integration'
install_url = f"git+https://github.com/{FORK}/PyHealth.git@{BRANCH}"
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", install_url,
     "--quiet", "--no-cache-dir", "--force-reinstall"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("PyHealth installation failed — see error above.")
print(f"✓ PyHealth installed from {FORK}/{BRANCH}")

# Environment detection — MUST come before any google.colab import
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import json
import glob
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from IPython.display import display

print(f"Running in Colab: {IN_COLAB}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("  → Runtime → Change runtime type → T4 GPU")

# Verify HuggingFace transformers version (>=4.48.3 required for use_cpu parameter)
import transformers
_ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert _ver >= (4, 48), (
    f"transformers>=4.48.3 required (got {transformers.__version__}). "
    "Fix: pip install transformers --upgrade"
)
print(f"transformers: {transformers.__version__} ✓")
print("✓ All setup complete")

---
# 2. Configuration

Configure all parameters here. **This is the only cell you need to modify.**

- **`PRESET = "demo"`** — 5 epochs, 1 K synthetic patients, ~30–45 min on T4
- **`PRESET = "production"`** — 20 epochs, 10 K synthetic patients, ~3–5 hrs on T4

In [ ]:
# ============================================================
# CONFIGURATION — All modifiable parameters in one place
# ============================================================

# --- Preset ---
PRESET = "demo"   # "demo" or "production"

# --- Training parameters ---
if PRESET == "demo":
    EPOCHS               = 5
    BATCH_SIZE           = 16
    N_SYNTHETIC_SAMPLES  = 1_000
    WARMUP_STEPS         = 100
elif PRESET == "production":
    EPOCHS               = 20
    BATCH_SIZE           = 16
    N_SYNTHETIC_SAMPLES  = 10_000
    WARMUP_STEPS         = 1_000

LR             = 1e-5   # Paper LR; low to avoid catastrophic forgetting of BART weights
MAX_SEQ_LENGTH = 512    # Max tokens per patient (visits + special tokens)

# --- Model architecture ---
D_HIDDEN      = 128  # Hidden dim for demographic prompt encoder
PROMPT_LENGTH = 1    # Prompt vectors per demographic feature (1 is sufficient per paper)

# --- BART backbone ---
# "facebook/bart-base": pretrained BART (139 M params, 768 hidden dim).
# PromptEHR fine-tunes these weights rather than training from scratch —
# the pretrained sequence modeling prior means even 20 epochs can produce good results.
BART_CONFIG_NAME = "facebook/bart-base"

# --- Generation parameters ---
RANDOM_SAMPLING = True   # True: nucleus sampling (diverse), False: greedy (deterministic)
TEMPERATURE     = 0.7    # Lower = more common codes. Higher = more rare/diverse codes.
TOP_P           = 0.95   # Nucleus sampling: sample from top 95% probability mass.

# --- Reproducibility ---
SEED = 42

# --- Paths (all derived from BASE_DIR) ---
BASE_DIR       = '/content/drive/MyDrive/PromptEHR_Training' if IN_COLAB else './promptehr_training'
DATA_DIR       = f'{BASE_DIR}/data'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
OUTPUT_DIR     = f'{BASE_DIR}/output'

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Preset:         {PRESET}")
print(f"Epochs:         {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LR}")
print(f"Synthetic:      {N_SYNTHETIC_SAMPLES:,} patients")
print(f"Base directory: {BASE_DIR}")
print("✓ Configuration complete")

---
# 3. Data Upload

Upload your MIMIC-III CSV files. PromptEHR needs **3 files** (one more than HALO — `PATIENTS.csv` is required for demographic conditioning):

1. `PATIENTS.csv` — date of birth and gender
2. `ADMISSIONS.csv` — admission timestamps (used to compute age at first admission)
3. `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

Files persist across Colab sessions when saved to Google Drive.

**No MIMIC-III?** The next cell automatically activates Demo Mode.

In [ ]:
import shutil
DEMO_MODE = False

# Mount Drive (Colab only) — guard makes this cell idempotent (safe to re-run)
if IN_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print("Drive already mounted")
    print("✓ Google Drive mounted")

# Check which files exist in the Drive-backed DATA_DIR
required_files = {
    'PATIENTS.csv':      'Patient demographics (DOB, gender)',
    'ADMISSIONS.csv':    'Admission records (timestamps)',
    'DIAGNOSES_ICD.csv': 'ICD-9 diagnosis codes',
}
existing = {f: os.path.exists(f'{DATA_DIR}/{f}') for f in required_files}
missing  = [f for f, ok in existing.items() if not ok]

if not missing:
    # All files already in Drive — no upload needed
    print("✓ All MIMIC-III files found in Drive (no upload needed):")
    for fname in required_files:
        size_mb = os.path.getsize(f'{DATA_DIR}/{fname}') / 1024 / 1024
        print(f"    {fname}  ({size_mb:.1f} MB)")
    print(f"\nFiles are reused from: {DATA_DIR}")
    print("To force re-upload, delete files from that folder and re-run this cell.")
else:
    print("MIMIC-III file status:")
    for fname, desc in required_files.items():
        mark = "✓" if existing[fname] else "✗ MISSING"
        print(f"  {mark}  {fname} — {desc}")

    if IN_COLAB:
        print(f"\nUploading {len(missing)} missing file(s)...")
        from google.colab import files as _colab_files
        uploaded = _colab_files.upload()

        # Normalize filenames — Colab renames duplicates as "ADMISSIONS (1).csv".
        # Match each upload to the required file it belongs to, then copy with
        # the canonical name so subsequent runs find the file in Drive.
        for uploaded_name, data in uploaded.items():
            matched = None
            for req in required_files:
                base = req.replace('.csv', '')
                if base in uploaded_name and uploaded_name.endswith('.csv'):
                    matched = req
                    break
            if matched:
                # Write upload bytes to /content/ then copy to Drive-backed dest
                tmp = f'/content/{uploaded_name}'
                with open(tmp, 'wb') as f:
                    f.write(data)
                dest = f'{DATA_DIR}/{matched}'
                shutil.copy(tmp, dest)
                size_mb = os.path.getsize(dest) / 1024 / 1024
                print(f"  ✓ Saved {matched} ({size_mb:.1f} MB) → {dest}")
            else:
                print(f"  ⚠  Unrecognised file: {uploaded_name} (skipped)")

        missing = [f for f in required_files if not os.path.exists(f'{DATA_DIR}/{f}')]

    if missing:
        print(f"\nMIMIC-III files not available ({missing}).")
        print("→ Activating Demo Mode — full pipeline with synthetic stand-in data.")
        DEMO_MODE = True
    else:
        print("\n✓ All 3 MIMIC-III files present. Running in MIMIC-III mode.")

In [ ]:
if DEMO_MODE:
    print("Setting up Demo Mode data...")
    from pyhealth.datasets.sample_dataset import InMemorySampleDataset

    # Synthetic stand-in: 200 patients, 2-6 visits, realistic ICD-9 codes.
    # Exercises the full pipeline without any real patient data.
    random.seed(SEED)
    icd9_pool = [
        "428.0", "401.9", "250.00", "272.4", "410.71", "486",
        "585.3", "V58.61", "412",   "414.01", "276.1", "285.9",
        "584.9", "305.1", "290.0",  "427.31", "518.81", "496",
        "038.9", "599.0",
    ]
    demo_samples = []
    for i in range(200):
        n_visits = random.randint(2, 6)
        visits   = [random.sample(icd9_pool, random.randint(1, 5)) for _ in range(n_visits)]
        demo_samples.append({
            "patient_id": f"DEMO_{i:04d}",
            "visits":     visits,
            "age":        float(random.randint(18, 89)),
            "gender":     random.randint(0, 1),
        })
    print(f"✓ Demo dataset: {len(demo_samples)} patients, up to 6 visits each")
    print("  (Replace with real MIMIC-III data for publication-quality results)")

In [ ]:
if not DEMO_MODE:
    print("Validating MIMIC-III files...")
    _patients = pd.read_csv(f'{DATA_DIR}/PATIENTS.csv')
    assert 'SUBJECT_ID' in _patients.columns, "PATIENTS.csv missing SUBJECT_ID"
    assert 'GENDER'     in _patients.columns, "PATIENTS.csv missing GENDER"
    assert 'DOB'        in _patients.columns, "PATIENTS.csv missing DOB"
    print(f"✓ PATIENTS.csv:      {len(_patients):>8,} rows")

    _admissions = pd.read_csv(f'{DATA_DIR}/ADMISSIONS.csv')
    assert 'SUBJECT_ID' in _admissions.columns, "ADMISSIONS.csv missing SUBJECT_ID"
    assert 'HADM_ID'    in _admissions.columns, "ADMISSIONS.csv missing HADM_ID"
    print(f"✓ ADMISSIONS.csv:    {len(_admissions):>8,} rows")

    _diagnoses = pd.read_csv(f'{DATA_DIR}/DIAGNOSES_ICD.csv')
    assert 'ICD9_CODE'  in _diagnoses.columns, "DIAGNOSES_ICD.csv missing ICD9_CODE"
    print(f"✓ DIAGNOSES_ICD.csv: {len(_diagnoses):>8,} rows")

    del _patients, _admissions, _diagnoses  # free memory
    print("\n✓ All files validated successfully")

---
# 4. Training

**What happens during training:**

1. **Dataset loading**: PyHealth reads MIMIC-III and creates one sample per patient (nested visit sequences + demographics: age at first admission, gender).
2. **Tokenization**: Each ICD-9 code is mapped to a unique BART token ID. Special tokens mark visit boundaries: `[VISIT_START]`, `[VISIT_END]`, `[SEQ_END]`.
3. **Demographic prompts**: Age and gender are encoded into learned prompt vectors prepended to the BART encoder input — steering the model toward age/gender-appropriate diagnosis patterns.
4. **Fine-tuning**: HuggingFace Trainer fine-tunes the BART Seq2Seq model to predict the next token conditioned on the demographic prompts.
5. **Checkpoint**: Saved to `{CHECKPOINT_DIR}/checkpoint.pt` after training.

The `WARMUP_STEPS` ramp up the learning rate gradually during early training, preventing catastrophic forgetting of BART's pretrained sequence modeling capabilities.

In [ ]:
# Set all random seeds before any stochastic operation
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
print(f"✓ Random seed set to {SEED}")

from pyhealth.datasets import split_by_patient
from pyhealth.models import PromptEHR

if not DEMO_MODE:
    from pyhealth.datasets import MIMIC3Dataset
    from pyhealth.tasks import promptehr_generation_mimic3_fn

    print("\nLoading MIMIC-III dataset (this may take a few minutes)...")
    dataset = MIMIC3Dataset(
        root=DATA_DIR,
        tables=["patients", "admissions", "diagnoses_icd"],
        code_mapping={},
    )
    print(f"Loaded {len(dataset.patients):,} patients")

    print("Applying PromptEHR generation task...")
    sample_dataset = dataset.set_task(promptehr_generation_mimic3_fn)
    print(f"Eligible patients (≥2 visits with ICD-9 codes): {len(sample_dataset):,}")
else:
    from pyhealth.datasets.sample_dataset import InMemorySampleDataset
    sample_dataset = InMemorySampleDataset(
        samples=demo_samples,
        input_schema={"visits": "nested_sequence"},
        output_schema={},
    )
    print(f"Demo dataset ready: {len(sample_dataset)} patients")

train_dataset, val_dataset, _ = split_by_patient(sample_dataset, [0.8, 0.1, 0.1])
print(f"\nSplit: {len(train_dataset):,} train / {len(val_dataset):,} val patients")

In [ ]:
# Save config alongside checkpoint for reproducibility
_config = {k: str(v) for k, v in globals().items()
           if k.isupper() and not k.startswith('_')
           and isinstance(v, (str, int, float, bool))}
_config['timestamp'] = datetime.now().isoformat()
_config_path = f'{CHECKPOINT_DIR}/config.json'
with open(_config_path, 'w') as f:
    json.dump(_config, f, indent=2)
print(f"✓ Config saved to {_config_path}")

# Initialize model
print("\nInitializing PromptEHR model...")
model = PromptEHR(
    dataset=train_dataset,
    n_num_features=1,        # 1 continuous demographic feature: age
    cat_cardinalities=[2],   # 1 categorical feature: gender (binary: 0=male, 1=female)
    d_hidden=D_HIDDEN,
    prompt_length=PROMPT_LENGTH,
    bart_config_name=BART_CONFIG_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    warmup_steps=WARMUP_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
    save_dir=CHECKPOINT_DIR,
)

n_special = 7  # PAD, BOS, EOS, UNK, VISIT_START, VISIT_END, SEQ_END
n_codes   = model._vocab.total_size - n_special
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ PromptEHR initialized")
print(f"  Vocabulary: {model._vocab.total_size} tokens "
      f"({n_codes} ICD-9 codes + {n_special} special tokens)")
print(f"  Parameters: {total_params:,}")

In [ ]:
print("Starting training...")
print("HuggingFace Trainer will print step-by-step progress below.")
print("=" * 60)

model.train_model(train_dataset, val_dataset=val_dataset)

print("=" * 60)
print("✓ Training complete!")
print(f"  Checkpoint: {CHECKPOINT_DIR}/checkpoint.pt")

In [ ]:
# Plot training loss from HuggingFace Trainer logs
_state_files = glob.glob(f'{CHECKPOINT_DIR}/**/trainer_state.json', recursive=True)

if _state_files:
    with open(_state_files[0]) as f:
        _log = json.load(f)['log_history']
    _steps  = [e['step'] for e in _log if 'loss' in e]
    _losses = [e['loss'] for e in _log if 'loss' in e]

    if _steps:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(_steps, _losses, 'b-o', linewidth=1.5, markersize=4, label='Training loss')
        ax.set_xlabel('Training step', fontsize=12)
        ax.set_ylabel('Cross-entropy loss', fontsize=12)
        ax.set_title('PromptEHR Training Loss', fontsize=14)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        _loss_plot = f'{OUTPUT_DIR}/training_loss.png'
        plt.savefig(_loss_plot, dpi=150); plt.show()
        print(f"Initial loss: {_losses[0]:.4f}  →  Final loss: {_losses[-1]:.4f}")
        print(f"Plot saved to: {_loss_plot}")
    else:
        print("No loss values recorded (too few steps for demo preset).")
else:
    print("trainer_state.json not found — skipping loss curve.")
    print("(Expected for very short demo runs.)")

---
# 5. Generation

**How generation works:**

1. **Demographic sampling**: For each synthetic patient, `synthesize_dataset` draws an `(age, gender)` pair from `model._demo_pool` — the real training population. This ensures the synthetic cohort's demographic profile mirrors MIMIC-III.
2. **Prompt conditioning**: The sampled demographics are encoded into prompt vectors and prepended to the BART encoder input.
3. **Autoregressive decoding**: BART generates tokens one at a time. Special tokens `[VISIT_START]` and `[VISIT_END]` structure the output into visits; `[SEQ_END]` ends the patient sequence.
4. **Decoding**: Token IDs are mapped back to ICD-9 code strings.

`RANDOM_SAMPLING = True` (default): nucleus sampling — diverse, realistic output.  
`RANDOM_SAMPLING = False`: greedy decoding — deterministic, may repeat common patterns.

In [ ]:
print(f"Generating {N_SYNTHETIC_SAMPLES:,} synthetic patients...")
print(f"  Sampling: {'nucleus (random)' if RANDOM_SAMPLING else 'greedy'}"
      + (f", temperature={TEMPERATURE}, top_p={TOP_P}" if RANDOM_SAMPLING else ""))
print("(This may take several minutes...)")

synthetic = model.synthesize_dataset(
    num_samples=N_SYNTHETIC_SAMPLES,
    random_sampling=RANDOM_SAMPLING,
)

print(f"\n✓ Generated {len(synthetic):,} synthetic patients")

# Preview
_preview = []
for p in synthetic[:10]:
    _v0 = p["visits"][0] if p["visits"] else []
    _sample = ", ".join(_v0[:4]) + ("..." if len(_v0) > 4 else "")
    _preview.append({
        "patient_id":       p["patient_id"],
        "n_visits":         len(p["visits"]),
        "total_codes":      sum(len(v) for v in p["visits"]),
        "first_visit_codes": _sample or "(empty)",
    })
display(pd.DataFrame(_preview))

In [ ]:
# Save as JSON (full nested records — directly loadable back into PyHealth)
json_path = f'{OUTPUT_DIR}/synthetic_patients.json'
with open(json_path, 'w') as f:
    json.dump(synthetic, f, indent=2)
print(f"✓ {len(synthetic):,} patients → {json_path}")

# Save as CSV (flat SUBJECT_ID, VISIT_NUM, ICD9_CODE — matches MIMIC-III output schema)
_rows = []
for p in synthetic:
    for _vnum, _visit in enumerate(p["visits"], 1):
        for _code in _visit:
            _rows.append({"SUBJECT_ID": p["patient_id"],
                          "VISIT_NUM":  _vnum,
                          "ICD9_CODE":  _code})
df_synthetic = pd.DataFrame(_rows)
csv_path = f'{OUTPUT_DIR}/synthetic_patients.csv'
df_synthetic.to_csv(csv_path, index=False)
print(f"✓ {len(df_synthetic):,} records → {csv_path}")
print(f"  Columns: SUBJECT_ID, VISIT_NUM, ICD9_CODE")
print("\nSample rows:")
display(df_synthetic.head(8))

---
# 6. Results & Evaluation

In [ ]:
print("=" * 60)
print("SYNTHETIC DATASET STATISTICS")
print("=" * 60)

n_visits = [len(p["visits"]) for p in synthetic]
n_codes  = [sum(len(v) for v in p["visits"]) for p in synthetic]

print(f"\nPatients:  {len(synthetic):,}")
print(f"\nVisits per patient:")
print(f"  Mean ± SD : {np.mean(n_visits):.2f} ± {np.std(n_visits):.2f}")
print(f"  Median    : {np.median(n_visits):.0f}")
print(f"  Range     : [{min(n_visits)}, {max(n_visits)}]")
print(f"\nDiagnosis codes per patient:")
print(f"  Mean ± SD : {np.mean(n_codes):.2f} ± {np.std(n_codes):.2f}")
print(f"  Median    : {np.median(n_codes):.0f}")
print(f"  Range     : [{min(n_codes)}, {max(n_codes)}]")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(n_visits, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax1.set_xlabel('Visits per patient'); ax1.set_ylabel('Count')
ax1.set_title('Visit Count Distribution')
ax2.hist(n_codes, bins=30, color='coral', edgecolor='white', alpha=0.85)
ax2.set_xlabel('Codes per patient'); ax2.set_ylabel('Count')
ax2.set_title('Code Count Distribution')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/count_distributions.png', dpi=150)
plt.show()

In [ ]:
all_synth_codes = set(c for p in synthetic for v in p["visits"] for c in v)
n_real_codes    = len(model._vocab._bart_to_code)  # ICD-9 codes in vocabulary
coverage        = len(all_synth_codes) / n_real_codes * 100 if n_real_codes > 0 else 0.0

print(f"Vocabulary size (ICD-9 codes): {n_real_codes:,}")
print(f"Unique codes in synthetic:     {len(all_synth_codes):,}")
print(f"Vocabulary coverage:           {coverage:.1f}%")

if coverage < 30:
    print("\n⚠  Low coverage may indicate mode collapse.")
    print("   Consider: more EPOCHS, lower LR, or check _demo_pool is populated.")
elif coverage < 60:
    print("\nModerate coverage — expected for demo preset.")
    print("Production training typically achieves 60–80%.")
else:
    print(f"\n✓ Good vocabulary coverage.")

In [ ]:
# model._demo_pool stores (age, gender) pairs from training data.
# synthesize_dataset samples from this pool for each synthetic patient,
# so the synthetic cohort's demographics automatically mirror the training population.
if model._demo_pool:
    _ages    = [a for a, g in model._demo_pool]
    _genders = [g for a, g in model._demo_pool]
    _n_male   = sum(1 for g in _genders if g == 0)
    _n_female = sum(1 for g in _genders if g == 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.hist(_ages, bins=25, density=True, color='steelblue', edgecolor='white',
             alpha=0.8, label='Training population')
    ax1.axvline(np.mean(_ages), color='navy', linestyle='--', linewidth=1.5,
                label=f'Mean age: {np.mean(_ages):.1f}')
    ax1.set_xlabel('Age at first admission', fontsize=12)
    ax1.set_ylabel('Density', fontsize=12)
    ax1.set_title('Age Distribution\n(Conditioning Source)', fontsize=13)
    ax1.legend(fontsize=10)

    _bars = ax2.bar(['Male', 'Female'], [_n_male, _n_female],
                    color=['steelblue', 'coral'], edgecolor='white', alpha=0.85)
    for _bar, _val in zip(_bars, [_n_male, _n_female]):
        ax2.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 5,
                 f'{_val:,}\n({_val/len(_genders)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11)
    ax2.set_ylabel('Patient count', fontsize=12)
    ax2.set_title('Gender Distribution\n(Conditioning Source)', fontsize=13)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/demographics_distribution.png', dpi=150)
    plt.show()

    print(f"Demographics pool: {len(model._demo_pool):,} training patients")
    print(f"  Age:    mean={np.mean(_ages):.1f}, std={np.std(_ages):.1f}, "
          f"range=[{min(_ages):.0f}, {max(_ages):.0f}]")
    print(f"  Male:   {_n_male:,} ({_n_male/len(_genders)*100:.1f}%)")
    print(f"  Female: {_n_female:,} ({_n_female/len(_genders)*100:.1f}%)")
    print("\n✓ Synthetic patients are generated with demographics sampled from this distribution.")
else:
    print("_demo_pool is empty — model was not trained before calling synthesize_dataset.")
    print("Run Section 4 first, or load a checkpoint that was saved after training.")

In [ ]:
# Build real training code frequencies by decoding processor-encoded visit tensors.
# NestedSequenceProcessor: index 0=pad, 1=unk, 2+=codes.
# _PromptEHRVocab mapping: bart_id = processor_idx + 5 for codes (idx>=2).
_vocab_map     = model._vocab._bart_to_code   # bart_token_id -> ICD-9 code string
_real_counts   = Counter()

for _sample in train_dataset:
    for _visit in _sample.get("visits", []):
        for _tok in _visit:
            _idx = int(_tok.item()) if hasattr(_tok, 'item') else int(_tok)
            if _idx >= 2:                              # skip pad(0) and unk(1)
                _bart_id = _idx + 5
                _code = _vocab_map.get(_bart_id)
                if _code:
                    _real_counts[_code] += 1

_synth_counts = Counter(c for p in synthetic for v in p["visits"] for c in v)

_top_codes  = [c for c, _ in _real_counts.most_common(20)]
_real_freq  = [_real_counts[c]         for c in _top_codes]
_synth_freq = [_synth_counts.get(c, 0) for c in _top_codes]

fig, ax = plt.subplots(figsize=(15, 5))
_x = range(len(_top_codes))
ax.bar([i - 0.2 for i in _x], _real_freq,  0.38, label='Real (training)', color='steelblue', alpha=0.85)
ax.bar([i + 0.2 for i in _x], _synth_freq, 0.38, label='Synthetic',       color='coral',     alpha=0.85)
ax.set_xticks(_x)
ax.set_xticklabels(_top_codes, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Top-20 ICD-9 Code Frequency: Real vs Synthetic', fontsize=14)
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/code_frequency_comparison.png', dpi=150)
plt.show()

# Pearson r (manual computation — no scipy dependency)
_r_mean = np.mean(_real_freq);  _s_mean = np.mean(_synth_freq)
_num    = sum((r - _r_mean)*(s - _s_mean) for r, s in zip(_real_freq, _synth_freq))
_denom  = (sum((r-_r_mean)**2 for r in _real_freq) * sum((s-_s_mean)**2 for s in _synth_freq)) ** 0.5
pearson_r = _num / _denom if _denom > 0 else 0.0
print(f"Pearson r (top-20 code frequencies, real vs synthetic): {pearson_r:.3f}")
if pearson_r > 0.8:   print("✓ Strong correlation — good distributional fidelity.")
elif pearson_r > 0.5: print("Moderate correlation — consider more epochs.")
else:                 print("Weak correlation — model may need more training.")

In [ ]:
_empty = [p for p in synthetic if not p["visits"] or all(len(v) == 0 for v in p["visits"])]
if _empty:
    print(f"⚠  {len(_empty)} / {len(synthetic)} patients have empty visit sequences.")
    print("   Possible causes:")
    print("   - Model is undertrained (increase EPOCHS)")
    print("   - Temperature too low (try TEMPERATURE = 1.0)")
    print("   - _demo_pool not populated (train before calling synthesize_dataset)")
else:
    print(f"✓ All {len(synthetic):,} patients have at least one visit with at least one code.")

In [ ]:
quality = {
    "total_synthetic_patients":  len(synthetic),
    "mean_visits_per_patient":   round(float(np.mean(n_visits)), 3),
    "std_visits_per_patient":    round(float(np.std(n_visits)),  3),
    "mean_codes_per_patient":    round(float(np.mean(n_codes)),  3),
    "std_codes_per_patient":     round(float(np.std(n_codes)),   3),
    "unique_codes_generated":    len(all_synth_codes),
    "vocabulary_size":           n_real_codes,
    "vocabulary_coverage_pct":   round(coverage, 2),
    "empty_patients_count":      len(_empty),
    "code_freq_pearson_r":       round(pearson_r, 4),
    "training_patients":         len(train_dataset),
    "vocab_total_size":          model._vocab.total_size,
    "demo_mode":                 DEMO_MODE,
    "preset":                    PRESET,
    "epochs":                    EPOCHS,
    "seed":                      SEED,
    "timestamp":                 datetime.now().isoformat(),
}
report_path = f'{OUTPUT_DIR}/quality_report.json'
with open(report_path, 'w') as f:
    json.dump(quality, f, indent=2)
print("Quality Report:")
print(json.dumps(quality, indent=2))
print(f"\n✓ Saved to {report_path}")

---
# 7. Download & Next Steps

In [ ]:
# Download output files (Colab only — silently skipped in local/SLURM environments)
_outputs = [
    csv_path,
    json_path,
    report_path,
    f'{OUTPUT_DIR}/training_loss.png',
    f'{OUTPUT_DIR}/demographics_distribution.png',
    f'{OUTPUT_DIR}/code_frequency_comparison.png',
    f'{CHECKPOINT_DIR}/checkpoint.pt',
    f'{CHECKPOINT_DIR}/config.json',
]

if IN_COLAB:
    from google.colab import files as _colab_files
    print("Downloading output files...")
    for _p in _outputs:
        if os.path.exists(_p):
            _colab_files.download(_p)
            print(f"  ✓ {os.path.basename(_p)}")
        else:
            print(f"  — {os.path.basename(_p)} (not found)")
else:
    print(f"Output files saved to: {OUTPUT_DIR}")
    print(f"Checkpoint:            {CHECKPOINT_DIR}/checkpoint.pt")
    for _p in _outputs:
        if os.path.exists(_p):
            _kb = os.path.getsize(_p) / 1024
            print(f"  {os.path.basename(_p):45s} {_kb:8.1f} KB")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINT RESUME — Run this cell instead of Section 4 if you already trained
# ─────────────────────────────────────────────────────────────────────────────
# Uncomment everything below to load an existing checkpoint, then skip to Section 5.

# from pyhealth.datasets import MIMIC3Dataset, split_by_patient
# from pyhealth.tasks import promptehr_generation_mimic3_fn
# from pyhealth.models import PromptEHR
#
# dataset = MIMIC3Dataset(
#     root=DATA_DIR,
#     tables=["patients", "admissions", "diagnoses_icd"],
#     code_mapping={},
# )
# sample_dataset = dataset.set_task(promptehr_generation_mimic3_fn)
# train_dataset, val_dataset, _ = split_by_patient(sample_dataset, [0.8, 0.1, 0.1])
#
# model = PromptEHR(
#     dataset=train_dataset,
#     n_num_features=1, cat_cardinalities=[2],
#     d_hidden=D_HIDDEN, prompt_length=PROMPT_LENGTH,
#     bart_config_name=BART_CONFIG_NAME,
#     epochs=EPOCHS, batch_size=BATCH_SIZE,
#     lr=LR, warmup_steps=WARMUP_STEPS,
#     max_seq_length=MAX_SEQ_LENGTH,
#     save_dir=CHECKPOINT_DIR,
# )
# ckpt = f'{CHECKPOINT_DIR}/checkpoint.pt'
# model.load_model(ckpt)
# print(f"✓ Loaded checkpoint from {ckpt}. Proceed to Section 5.")

print("(Resume template — uncomment the lines above to use)")

---
## 🎉 Congratulations!

You've successfully:
1. ✅ Trained a PromptEHR model conditioned on patient demographics
2. ✅ Generated synthetic patients whose age/gender distribution mirrors MIMIC-III
3. ✅ Validated ICD-9 code frequency fidelity against real training data
4. ✅ Saved output files for downstream use

## Next Steps

**Use your synthetic data:**
- Train readmission/mortality/LoS prediction models on synthetic data
- Evaluate fairness across demographic subgroups
- Share synthetic patients without privacy concerns

**Reload and generate more:**
```python
from pyhealth.models import PromptEHR
model = PromptEHR(dataset=train_dataset, ...)
model.load_model('./promptehr_training/checkpoints/checkpoint.pt')
extra = model.synthesize_dataset(num_samples=50_000)
```

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `AssertionError: transformers>=4.48.3 required` | Old transformers installed | `pip install transformers --upgrade` |
| Empty patients in output | Undertrained model | Increase `EPOCHS` or raise `TEMPERATURE` to `1.0` |
| Training loss not decreasing after 2+ epochs | LR too high | Try `LR = 5e-6` and `WARMUP_STEPS = 500` |
| Out of memory (OOM) | Batch too large | Reduce `BATCH_SIZE = 8` |
| Very slow training | No GPU | Runtime → Change runtime type → T4 GPU |
| `KeyError: 'visits'` in demo mode | Wrong schema | Ensure `input_schema={"visits": "nested_sequence"}` |
| Synthetic codes all the same | Temperature too low | Try `TEMPERATURE = 1.0`, `RANDOM_SAMPLING = True` |

---

## Reference

Wang, Y., et al. "PromptEHR: Conditional Electronic Healthcare Records Generation with Prompt Learning." *EMNLP 2023*. https://arxiv.org/abs/2211.01761

---
_Notebook for PyHealth 2.0 · Branch: `promptehr-pr-integration` · jalengg/PyHealth_